# Feature-value audit: do our engineered signals address Qwen’s errors?

No new fitting, neural inference, ensemble, or submission. The 881-comment cohort is repeatedly inspected development data—not hidden Kaggle output or independent confirmation.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/feature_value_audit.json").is_file())
from scripts.run_feature_value_audit import figures
r = json.loads((ROOT / "reports/feature_value_audit/audit.json").read_text())
print("Audit:", r["run_id"], "| Fits:", r["model_fits"], "| Neural calls:", r["new_neural_inference"])
CHARTS = figures(r)

Audit: a6103241df718b4b27a4 | Fits: 0 | Neural calls: 0


## 1. Round 5’s matched control matters

The full context representation improved on the smaller anchor but slightly trailed its matched copy. Inspect the five-round ledger without promoting adaptive anchors.

In [2]:
display(pd.DataFrame(r["research_ledger"]))
CHARTS[6].show(renderer="plotly_mimetype")

,round,family,comparison,delta_auc,simultaneous_low,simultaneous_high,valid_draws,historical_decision,candidate,reference
0,1,Behavior addition,add_behavior vs control,0.022354,-0.016498,0.061205,500,DO_NOT_PROMOTE_THIS_FEATURE_SET,NaN,NaN
1,2,Actor-action addition,add_act_roles vs behavior,0.006238,-0.014359,0.026835,500,DO_NOT_PROMOTE_PRIMARY,NaN,NaN
2,3,Policy conditioning vs copy,conditioning: lexical,0.009704,-0.014177,0.033584,500,DO_NOT_PROMOTE_PRIMARY,condition_lexical,copy_lexical
3,4,Diagnostic lexical weighting vs shuffle,Learned weights versus permuted weights,0.007137,-0.002522,0.016796,500,DO_NOT_PROMOTE_PRIMARY,rule_both,permuted_both
4,5,Lexical scope vs copy,Scope versus collapsed: both,-0.000379,-0.008407,0.007650,500,DO_NOT_PROMOTE_PRIMARY,scope_both,copy_both


## 2. Same comments, verified model outputs

The Qwen comparator uses cached development-fold margins and reproduces the published per-policy AUC. This is not comparing a local 0.70 score with a Kaggle 0.91 score. Probability metrics use raw probabilities, not ranks.

In [3]:
display(pd.DataFrame(r["metrics"]))
CHARTS[0].show(renderer="plotly_mimetype")

,fold,policy,model,auc,brier,log_loss
0,0,"No Advertising: Spam, referral links, unsolici...",qwen_adapted_reference,0.679254,0.250238,0.760003
1,0,"No Advertising: Spam, referral links, unsolici...",lexical_control,0.673022,0.233599,0.667527
2,0,"No Advertising: Spam, referral links, unsolici...",add_behavior,0.675784,0.231577,0.663719
3,0,"No Advertising: Spam, referral links, unsolici...",add_act_roles,0.675784,0.231472,0.663380
4,0,"No Advertising: Spam, referral links, unsolici...",condition_lexical,0.686604,0.231335,0.669720
5,0,"No Advertising: Spam, referral links, unsolici...",rule_both,0.688694,0.231786,0.670448
6,0,"No Advertising: Spam, referral links, unsolici...",copy_both,0.695410,0.232222,0.678693
7,0,"No Advertising: Spam, referral links, unsolici...",scope_both,0.695485,0.232151,0.678278
8,1,No legal advice: Do not offer or request legal...,qwen_adapted_reference,0.760533,0.221528,0.709879
9,1,No legal advice: Do not offer or request legal...,lexical_control,0.641993,0.232466,0.656063


## 3. Uncertainty relative to Qwen

Seven fixed differences; normalized-comment-group paired bootstrap. Conditional intervals do not correct the full adaptive five-round history or establish independent replication.

In [4]:
display(pd.DataFrame(r["contrasts"]))
CHARTS[1].show(renderer="plotly_mimetype")

,model,delta_auc,simultaneous_low,simultaneous_high,valid_draws
0,lexical_control,-0.062386,-0.107250,-0.017521,500
1,add_behavior,-0.040032,-0.084896,0.004833,500
2,add_act_roles,-0.033794,-0.078659,0.011070,500
3,condition_lexical,-0.025375,-0.070239,0.019489,500
4,rule_both,-0.017647,-0.062512,0.027217,500
5,copy_both,-0.011417,-0.056282,0.033447,500
6,scope_both,-0.011796,-0.056660,0.033069,500


## 4. Ordering improvements and damage

AUC measures ranking of positive-negative pairs. Half credit is given to ties. Pairwise improvements minus damage equal the AUC change. Pairs share comments and are NOT independent samples. Correlation is descriptive; no blend is fitted.

In [5]:
display(pd.DataFrame(r["pair_decomposition"]))
CHARTS[2].show(renderer="plotly_mimetype")
CHARTS[3].show(renderer="plotly_mimetype")

,fold,policy,model,pairs,gained_auc_credit,lost_auc_credit,net_auc_delta,pairs_improved,pairs_worsened,reference_ties,candidate_ties,pair_count_is_not_independent_sample_size
0,0,"No Advertising: Spam, referral links, unsolici...",lexical_control,13400,0.140336,0.146567,-0.006231,1911,1985,104,1,True
1,0,"No Advertising: Spam, referral links, unsolici...",add_behavior,13400,0.149067,0.152537,-0.003470,2028,2065,104,1,True
2,0,"No Advertising: Spam, referral links, unsolici...",add_act_roles,13400,0.150821,0.154291,-0.003470,2052,2088,104,1,True
3,0,"No Advertising: Spam, referral links, unsolici...",condition_lexical,13400,0.149776,0.142425,0.007351,2037,1930,104,1,True
4,0,"No Advertising: Spam, referral links, unsolici...",rule_both,13400,0.146007,0.136567,0.009440,1987,1851,104,1,True
5,0,"No Advertising: Spam, referral links, unsolici...",copy_both,13400,0.145560,0.129403,0.016157,1982,1754,104,1,True
6,0,"No Advertising: Spam, referral links, unsolici...",scope_both,13400,0.147015,0.130784,0.016231,2001,1773,104,1,True
7,1,No legal advice: Do not offer or request legal...,lexical_control,102202,0.106397,0.224937,-0.118540,11013,23131,562,0,True
8,1,No legal advice: Do not offer or request legal...,add_behavior,102202,0.113168,0.189761,-0.076593,11720,19521,562,0,True
9,1,No legal advice: Do not offer or request legal...,add_act_roles,102202,0.119489,0.183607,-0.064118,12363,18895,562,0,True


## 5. Which fixed contexts lack reliable ranking?

Text flags are created without targets; labels are used only afterwards for descriptive scoring. Slices overlap and are approximate. Missing bars mean insufficient support, not AUC=0. Inspect present and absent subsets; never invent conversation context.

In [6]:
display(pd.DataFrame(r["slices"]))
CHARTS[4].show(renderer="plotly_mimetype")
CHARTS[5].show(renderer="plotly_mimetype")

,fold,policy,slice,present,rows,positive_rows,negative_rows,model,auc,status
0,0,"No Advertising: Spam, referral links, unsolici...",explicit_quotation,True,7,6,1,qwen_adapted_reference,NaN,insufficient_class_support
1,0,"No Advertising: Spam, referral links, unsolici...",explicit_quotation,True,7,6,1,rule_both,NaN,insufficient_class_support
2,0,"No Advertising: Spam, referral links, unsolici...",explicit_quotation,True,7,6,1,scope_both,NaN,insufficient_class_support
3,0,"No Advertising: Spam, referral links, unsolici...",explicit_quotation,False,227,128,99,qwen_adapted_reference,0.678938,descriptive_only
4,0,"No Advertising: Spam, referral links, unsolici...",explicit_quotation,False,227,128,99,rule_both,0.700245,descriptive_only
...,...,...,...,...,...,...,...,...,...,...
79,1,No legal advice: Do not offer or request legal...,first_person_request,True,7,3,4,rule_both,NaN,insufficient_class_support
80,1,No legal advice: Do not offer or request legal...,first_person_request,True,7,3,4,scope_both,NaN,insufficient_class_support
81,1,No legal advice: Do not offer or request legal...,first_person_request,False,640,370,270,qwen_adapted_reference,0.761522,descriptive_only
82,1,No legal advice: Do not offer or request legal...,first_person_request,False,640,370,270,rule_both,0.718198,descriptive_only


## 6. Probability quality is separate from ranking

Model-generated probabilities are not assumed calibrated. Do not interpret within-policy percentile ranks as probabilities.

In [7]:
CHARTS[7].show(renderer="plotly_mimetype")
print("Qwen cache:", r["identity"]["config"]["reference_run"])
print("Recovered inputs:", r["reference_recovery"])

Qwen cache: d13858b7407993fcc6e8
Recovered inputs: [{'fold': 0, 'origin': 'verified_private_s3_read', 'bytes': 21493371, 'sha256': 'd4413fed09cb18d330b655656d41143759c8012dcce3e8a4517029dda18eff8b'}, {'fold': 1, 'origin': 'verified_private_s3_read', 'bytes': 21359643, 'sha256': '39530ccecc3cfc1f0dbd7d222ab6d02dcbe84944c0ceff27d46f8826e99622c2'}]


## 7. Decision boundary

This is a diagnostic selection of the next feature hypothesis, not promotion or an ensemble experiment. Stop blind scope expansion after a matched-control failure. Use residual ranking evidence to choose one rule-conditioned representation family. Read `docs/FEATURE_VALUE_AUDIT.md` before another model run.

In [8]:
print("Promotion:", r["promotion"])
print("Next gate:", r["next_gate"])
for item in r["limitations"]:
    print("—", item)
print("Dashboard:", ROOT / "reports/feature_value_audit/dashboard.html")

Promotion: NONE_DIAGNOSTIC_ONLY
Next gate: choose_one_preregistered_feature_hypothesis_after_review
— Cached Qwen scores are from development folds, not hidden Kaggle predictions.
— The 881 comments have been repeatedly inspected; this is not fresh validation.
— Conditional intervals cover seven contrasts, not all historical selection.
— Pairs share comments; pair counts are not independent sample sizes.
— Slices are fixed, overlapping, approximate text flags; not causal mechanisms.
— Ranking complementarity does not guarantee useful feature additions or blends.
— No threshold, calibrator, ensemble weight, or feature is fitted in this audit.
— Source/data hashes and exact historical row order are required before scoring.
Dashboard: /home/sagemaker-user/projects/jigsaw-rule-classifier/reports/feature_value_audit/dashboard.html
